Convert g.Nautilus (33ch) -> EEG_RAW_V1 (uV @160 Hz + events)

This notebook:
- Finds all raw `.hdf5` files (g.Nautilus 33ch) in `DATASET_DIR` (skipping EEG files already converted).
- Converts signals to uV @160 Hz with the pipeline: HP@500 -> resample 500->160 -> notch 50 -> BP 0.5-40 Hz.
- Rebuilds events from the trigger channel (bit 8 = Perform, bit 16 = Blank segment) using the PERFORM protocol schedule.
- Writes outputs in EEG_RAW_V1 format with:
  - `/raw/data` (32xS, uV @160 Hz), `/raw/ch_names`, `/raw/sfreq`, `/raw/units="uV"`.
  - `/events/table` (onset, duration, label: L/R/F/B/0).
  - Group `/norm` with `data_z` (z-score vs. EEGMMI_REF); note this z-norm is for analysis and is not used by inference/fine-tuning.
- Saves a `conversion_manifest.csv` manifest with a per-file summary (samples and events).


In [1]:
from pathlib import Path
# ================== PATHS ==================

DATASET_DIR = Path(r"path\to\dataset")              # folder containing raw .hdf5 files
EEGMMI_REF  = DATASET_DIR / "RIFERIMENTO.hdf5"      # reference file in EEG_RAW_V1 format
OUT_DIR     = DATASET_DIR / "EEG_RAW_V1_out"        # where to save converted files

# ================== PIPELINE PARAMETERS ==================
# Note: this pipeline matches training/inference (notch+BP at 160 Hz).
HP_CUTOFF        = 0.20       # Hz (0.20–0.25 ok)
FS_IN_DEFAULT    = 500.0      # Fallback Fs if not found in the raw file
FS_OUT           = 160.0
SCALE_UV_PER_UNIT= 2.722      # fattore µV/unit (stima g.Nautilus, adattare se necessario)

# Target channels (standard 10-10)
TARGET32 = [
    "Fp1","Fp2","AF3","AF4","F7","F3","Fz","F4","F8",
    "FC5","FC1","FC2","FC6","T7","C3","Cz","C4","T8",
    "CP5","CP1","CP2","CP6","P7","P3","Pz","P4","P8",
    "PO7","PO3","PO4","PO8","Oz"
]

# Schedule (PERFORM) — (index, name, group, duration_sec)
SCHEDULE_ROWS = [
(0,"Relax","Slides",8),(1,"Baseline","Slides",20),(2,"Gifs","Categories",5),(3,"FixationCross","Slides",5),
(4,"BothHands","Gifs",5),(5,"Perform","Slides",5),(6,"Blank","Slides",8),(7,"FixationCross","Slides",5),
(8,"RightHand","Gifs",5),(9,"Perform","Slides",5),(10,"Blank","Slides",8),(11,"FixationCross","Slides",5),
(12,"LeftHand","Gifs",5),(13,"Perform","Slides",5),(14,"Blank","Slides",8),(15,"FixationCross","Slides",5),
(16,"LiftFeet","Gifs",5),(17,"Perform","Slides",5),(18,"Blank","Slides",8),(19,"FixationCross","Slides",5),
(20,"LeftHand","Gifs",5),(21,"Perform","Slides",5),(22,"Blank","Slides",8),(23,"FixationCross","Slides",5),
(24,"LiftFeet","Gifs",5),(25,"Perform","Slides",5),(26,"Blank","Slides",8),(27,"FixationCross","Slides",5),
(28,"LiftFeet","Gifs",5),(29,"Perform","Slides",5),(30,"Blank","Slides",8),(31,"FixationCross","Slides",5),
(32,"BothHands","Gifs",5),(33,"Perform","Slides",5),(34,"Blank","Slides",8),(35,"FixationCross","Slides",5),
(36,"RightHand","Text",5),(37,"Perform","Slides",5),(38,"Blank","Slides",8),(39,"FixationCross","Slides",5),
(40,"LeftHand","Text",5),(41,"Perform","Slides",5),(42,"Blank","Slides",8),(43,"FixationCross","Slides",5),
(44,"RightHand","Text",5),(45,"Perform","Slides",5),(46,"Blank","Slides",8),(47,"FixationCross","Slides",5),
(48,"BothHands","Text",5),(49,"Perform","Slides",5),(50,"Blank","Slides",8),(51,"Text","Categories",5),
(52,"FixationCross","Slides",5),(53,"LiftFeet","Text",5),(54,"Perform","Slides",5),(55,"Blank","Slides",8),
(56,"FixationCross","Slides",5),(57,"LiftFeet","Text",5),(58,"Perform","Slides",5),(59,"Blank","Slides",8),
(60,"FixationCross","Slides",5),(61,"BothHands","Text",5),(62,"Perform","Slides",5),(63,"Blank","Slides",8),
(64,"FixationCross","Slides",5),(65,"BothHands","Text",5),(66,"Perform","Slides",5),(67,"Blank","Slides",8),
(68,"FixationCross","Slides",5),(69,"BothHands","Text",5),(70,"Perform","Slides",5),(71,"Blank","Slides",8),
(72,"FixationCross","Slides",5),(73,"RightHand","Text",5),(74,"Perform","Slides",5),(75,"Blank","Slides",8),
(76,"FixationCross","Slides",5),(77,"RightHand","Text",5),(78,"Perform","Slides",5),(79,"Blank","Slides",8),
(80,"FixationCross","Slides",5),(81,"LeftHand","Text",5),(82,"Perform","Slides",5),(83,"Blank","Slides",8),
(84,"FixationCross","Slides",5),(85,"RightHand","Text",5),(86,"Perform","Slides",5),(87,"Blank","Slides",8),
(88,"FixationCross","Slides",5),(89,"LeftHand","Text",5),(90,"Perform","Slides",5),(91,"Blank","Slides",8),
(92,"FixationCross","Slides",5),(93,"LeftHand","Text",5),(94,"Perform","Slides",5),(95,"Blank","Slides",8),
(96,"FixationCross","Slides",5),(97,"LiftFeet","Text",5),(98,"Perform","Slides",5),(99,"Blank","Slides",8),
(100,"Relax","Slides",5),(101,"Baseline","Slides",20)
]

# Anonymization map for subject names
NAME_MAP = {
    "Adrien": "Subject01",
    "Canesi": "Subject02",
    "Jules": "Subject03",
    "Luigi": "Subject04",
    "Marelli": "Subject05",
}

def anonymize_stem(stem: str) -> str:
    """Map personal stem to anonymized SubjectXX_Session_* form."""
    s = stem.replace("Sessione", "Session").replace("Parziale_(batteria_scarica)", "Partial_low_battery")
    parts = s.split("_")
    if not parts:
        return s
    parts[0] = NAME_MAP.get(parts[0], parts[0])
    return "_".join(parts)


In [2]:
import os, json, warnings
from typing import List, Tuple, Dict, Optional

import h5py, numpy as np, pandas as pd
from scipy.signal import butter, filtfilt, iirnotch, resample_poly
warnings.filterwarnings("ignore", category=RuntimeWarning)


In [3]:
def butter_hp(sig, fs, cutoff, order=4):
    # High-pass filter 
    b, a = butter(order, cutoff/(0.5*fs), btype='high')
    return filtfilt(b, a, sig, axis=0)  

def butter_bp(sig, fs, lo, hi, order=4):
    # Band-pass filter
    b, a = butter(order, [lo/(0.5*fs), hi/(0.5*fs)], btype='band')
    return filtfilt(b, a, sig, axis=0)

def apply_notch(sig, fs, f0=50.0, Q=30.0):
    # Notch filter (narrow-band suppression)
    # Q: quality factor (higher = narrower notch).
    b, a = iirnotch(w0=f0/(0.5*fs), Q=Q)
    return filtfilt(b, a, sig, axis=0)

def rising_edges(mask_bool: np.ndarray) -> np.ndarray:
    return np.where(np.logical_and(mask_bool, np.concatenate([[False], ~mask_bool[:-1]])))[0]

def _cue_seq_from_schedule(rows=SCHEDULE_ROWS):
    # Build the sequence of motor cues (Right/Left/Feet/Hands)
    # based on the experimental schedule.
    # Each time "Perform" appears, assign the last valid cue.
    seq, last = [], None
    for _, name, _, _ in rows:
        if name in {"RightHand","LeftHand","BothHands","LiftFeet"}:
            last = name
        if name == "Perform":
            seq.append(last)
    return seq

# Mapping from cue names to the compact labels used in the dataset.
CUE_TO_LABEL = {"RightHand":"R","LeftHand":"L","BothHands":"F","LiftFeet":"B"}


In [4]:
def load_raw_33(path: Path) -> tuple[np.ndarray, np.ndarray, float]:
    """
    Load raw g.Nautilus 33ch file:
      - X[:, :32] = EEG counts
      - X[:, 32]  = trigger
      - Fs: try multiple fields/notations
    Trim the portion between first/last bit64 if present.
    """
    with h5py.File(path, "r") as f:
        X = f["RawData"]["Samples"][()]  # (N,33)
        sr = None
        for cand in ["RawData/SamplingRate","RawData/SampleRate","RawData/Fs","SampleRate","SamplingRate"]:
            if cand in f:
                v = f[cand][()]; sr = float(v if np.isscalar(v) else v[0]); break
        if sr is None and "RawData" in f and isinstance(f["RawData"], h5py.Group):
            for a in ["SamplingRate","SampleRate","Fs","FS"]:
                if a in f["RawData"].attrs:
                    sr = float(f["RawData"].attrs[a]); break
    sr = sr or FS_IN_DEFAULT
    trig = X[:, 32].astype(np.int64)
    eeg  = X[:, :32].astype(np.float64)
    # trim between first/last bit 64
    m64 = (trig & 64) == 64
    if np.any(m64):
        idx = np.where(m64)[0]
        s0, s1 = int(idx.min()), int(idx.max()) + 1
        trig = trig[s0:s1]; eeg = eeg[s0:s1]
    return eeg, trig, float(sr)

def preprocess_counts_to_uV_160(eeg_counts: np.ndarray, fs_in: float) -> np.ndarray:
    """
    Pipeline: HP@500 ➜ resample 500→160 ➜ notch 50 (±100) ➜ BP 0.5–40 ➜ scala a µV
    Returns float32 (S160, 32).
    """
    X = butter_hp(eeg_counts, fs_in, HP_CUTOFF, order=4)
    X = resample_poly(X, up=8, down=25, axis=0)
    X = apply_notch(X, FS_OUT, 50.0, Q=30.0)
    if DO_NOTCH_100: 
        X = apply_notch(X, FS_OUT, 100.0, Q=30.0)
    X = butter_bp(X, FS_OUT, 0.5, 40.0, order=4)
    return (X * SCALE_UV_PER_UNIT).astype(np.float32)

def mu_sigma_from_eegmmi(eegmmi_path: Path) -> tuple[np.ndarray, np.ndarray]:
    """
    Calcola µ/σ su EEGMMI_REF (µV @160) con la stessa post-filt. (notch+BP) usata sopra,
    so the auxiliary z-norm stored in the converted file stays consistent.
    """
    with h5py.File(eegmmi_path, "r") as f:
        Y = f["/raw/data"][()].T         # (S,32) µV @160
        fs = float(f["/raw/sfreq"][()])
    Y = apply_notch(Y, fs, 50.0, Q=30.0)
    if DO_NOTCH_100:
        Y = apply_notch(Y, fs, 100.0, Q=30.0)
    Y = butter_bp(Y, fs, 0.5, 40.0, order=4)
    mu = np.mean(Y, axis=0).astype(np.float32)
    sd = np.std(Y, axis=0, ddof=0).astype(np.float32)
    sd[sd < 1e-8] = 1.0
    return mu, sd

def build_events_from_trigger(trig_500: np.ndarray, fs_in: float, n_out: int) -> np.ndarray:
    """
    Reconstruct PERFORM events from trigger (bit 8 = Perform, bit 16 = Blank).
    Output dtype strutturato: onset_sample, duration_samples, label (S1), meta (utf8).
    """
    cue_seq = _cue_seq_from_schedule()
    on_per = rising_edges((trig_500 & 8) == 8)
    on_blk = rising_edges((trig_500 & 16) == 16)
    nmap = min(len(on_per), len(cue_seq))
    on_per, cue_seq = on_per[:nmap], cue_seq[:nmap]

    def map_500_to_160(n): 
        return int(round(n * (FS_OUT / fs_in)))
    WIN4, WIN5 = int(4*FS_OUT), int(5*FS_OUT)

    rows = []
    # PERFORM (5s) labeled L/R/F/B
    for k, idx in enumerate(on_per):
        o = map_500_to_160(int(idx))
        if o < n_out:
            lab = CUE_TO_LABEL.get(cue_seq[k], None)
            if lab: 
                rows.append((o, WIN5, lab.encode("ascii"), f"PERFORM|cue={cue_seq[k]}"))
    # BLANK (two 4s segments, label '0')
    for idx in on_blk:
        o1 = map_500_to_160(int(idx)); o2 = o1 + WIN4
        if o1 < n_out: rows.append((o1, min(WIN4, n_out-o1), b"0", "BLANK|seg=1/2"))
        if o2 < n_out: rows.append((o2, min(WIN4, n_out-o2), b"0", "BLANK|seg=2/2"))
    rows.sort(key=lambda t: t[0])

    dt_label = np.dtype("S1")
    dt_meta  = h5py.string_dtype(encoding="utf-8")
    ev = np.zeros((len(rows),), dtype=np.dtype([
        ("onset_sample",np.int64),("duration_samples",np.int64),("label",dt_label),("meta",dt_meta)
    ]))
    for i,(o,d,lb,mt) in enumerate(rows):
        ev["onset_sample"][i] = int(o); ev["duration_samples"][i] = int(d)
        ev["label"][i] = lb; ev["meta"][i] = mt
    return ev

def convert_one(in_path: Path, out_path: Path, mu_ref: np.ndarray, sd_ref: np.ndarray) -> Dict:
    """
    Converte un file raw 33ch in EEG_RAW_V1 e ritorna un piccolo dizionario riassuntivo.
    """
    eeg_counts, trig, fs_in = load_raw_33(in_path)
    X_uV = preprocess_counts_to_uV_160(eeg_counts, fs_in)      # (S160,32)
    X_z  = (X_uV - mu_ref[None,:]) / sd_ref[None,:]
    ev   = build_events_from_trigger(trig, fs_in, X_uV.shape[0])

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with h5py.File(out_path, "w") as f:
        f.create_dataset("/meta/version", data=np.string_("EEG_RAW_V1"))
        grp = f.create_group("/raw")
        dt_utf8 = h5py.string_dtype(encoding="utf-8")
        grp.attrs.create("producer",  np.array("gNautilus->EEG_RAW_V1", dtype=dt_utf8))
        grp.attrs.create("reference", np.array("as_recorded", dtype=dt_utf8))
        grp.attrs.create("montage",   np.array("standard_10-10 (assumed)", dtype=dt_utf8))
        grp.attrs.create("scale_to_uV", np.array(1.0, dtype=np.float32))

        f.create_dataset("/raw/data",  data=X_uV.T.astype(np.float32))  # (32,S)
        f.create_dataset("/raw/sfreq", data=np.array(FS_OUT, dtype=np.float32))
        f.create_dataset("/raw/ch_names", data=np.array([np.string_(c) for c in TARGET32]))
        f.create_dataset("/raw/units", data=np.string_("uV"))
        f.create_dataset("/events/table", data=ev)

        ngrp = f.create_group("/norm")
        ngrp.create_dataset("data_z", data=X_z.T.astype(np.float32))
        ngrp.create_dataset("z_mean_uV", data=mu_ref.astype(np.float32))
        ngrp.create_dataset("z_std_uV",  data=sd_ref.astype(np.float32))
        ngrp.attrs.create("hp_cutoff_hz", np.array(HP_CUTOFF, dtype=np.float32))
        ngrp.attrs.create("filters", np.array("HP@500 -> resample 500->160 -> notch50 -> BP 0.5-40", dtype=dt_utf8))

    # event counts
    labels = [ (lb.decode() if isinstance(lb, (bytes,bytearray)) else str(lb)) for lb in ev["label"] ]
    counts = {k:int(np.sum(np.array(labels)==k)) for k in ["L","R","F","B","0"]}
    print(f"[OK] Converted -> {out_path.name}")
    return {
        "name": out_path.stem,
        "samples_160": int(X_uV.shape[0]),
        "n_events_total": int(len(ev)),
        "n_L": counts.get("L",0),
        "n_R": counts.get("R",0),
        "n_F": counts.get("F",0),
        "n_B": counts.get("B",0),
        "n_0": counts.get("0",0),
    }


In [5]:
# Find all raw .hdf5 files (not yet converted)
OUT_DIR.mkdir(parents=True, exist_ok=True)
all_h5 = sorted([p for p in DATASET_DIR.glob("*.hdf5")
                 if "EEG_RAW_V1" not in p.name and p.name.lower() != "protocollo.csv"])
anon_raw = [f"{anonymize_stem(p.stem)}.hdf5" for p in all_h5]
print("[INFO] Found raw files:", ", ".join(anon_raw) if anon_raw else "(none)")

# Reference mu/sigma (for /norm) computed on EEGMMI_REF with the same post-filter.
mu_ref, sd_ref = mu_sigma_from_eegmmi(EEGMMI_REF)
print("[INFO] mu/sigma EEGMMI computed.")

# Conversion
summary_rows = []
for p in all_h5:
    stem_out = anonymize_stem(p.stem)
    out_p = OUT_DIR / f"{stem_out}_EEG_RAW_V1.hdf5"
    try:
        row = convert_one(p, out_p, mu_ref, sd_ref)
        summary_rows.append(row)
    except Exception as e:
        print(f"[ERR ] Conversion {p.name}: {e}")

# Manifest
if summary_rows:
    manifest = pd.DataFrame(summary_rows).sort_values("name").reset_index(drop=True)
else:
    manifest = pd.DataFrame(columns=["name","samples_160","n_events_total","n_L","n_R","n_F","n_B","n_0"])

csv_manifest = OUT_DIR / "conversion_manifest.csv"
manifest.to_csv(csv_manifest, index=False, encoding="utf-8")
print(f">> Manifest saved to: {csv_manifest}")

try:
    display(manifest)
except Exception:
    print(manifest)


[INFO] Found raw files: Subject01_Session_1.hdf5, Subject01_Session_2.hdf5, Subject02_Session_1.hdf5, Subject02_Session_2.hdf5, Subject03_Session_1.hdf5, Subject03_Session_2.hdf5, Subject04_Session_1.hdf5, Subject04_Session_2.hdf5, Subject05_Session_1.hdf5, Subject05_Session_1_Partial_low_battery.hdf5, Subject05_Session_2.hdf5
[INFO] mu/sigma EEGMMI computed.
[OK] Converted -> Subject01_Session_1_EEG_RAW_V1.hdf5
[OK] Converted -> Subject01_Session_2_EEG_RAW_V1.hdf5
[OK] Converted -> Subject02_Session_1_EEG_RAW_V1.hdf5
[OK] Converted -> Subject02_Session_2_EEG_RAW_V1.hdf5
[OK] Converted -> Subject03_Session_1_EEG_RAW_V1.hdf5
[OK] Converted -> Subject03_Session_2_EEG_RAW_V1.hdf5
[OK] Converted -> Subject04_Session_1_EEG_RAW_V1.hdf5
[OK] Converted -> Subject04_Session_2_EEG_RAW_V1.hdf5
[OK] Converted -> Subject05_Session_1_EEG_RAW_V1.hdf5
[OK] Converted -> Subject05_Session_1_Partial_low_battery_EEG_RAW_V1.hdf5
[OK] Converted -> Subject05_Session_2_EEG_RAW_V1.hdf5
>> Manifest saved to: C:

,name,samples_160,n_events_total,n_L,n_R,n_F,n_B,n_0
0,Subject01_Session_1_EEG_RAW_V1,100929,72,6,6,6,6,48
1,Subject01_Session_2_EEG_RAW_V1,99891,72,6,6,6,6,48
2,Subject02_Session_1_EEG_RAW_V1,100452,72,6,6,6,6,48
3,Subject02_Session_2_EEG_RAW_V1,99371,72,6,6,6,6,48
4,Subject03_Session_1_EEG_RAW_V1,99979,72,6,6,6,6,48
5,Subject03_Session_2_EEG_RAW_V1,106082,72,6,6,6,6,48
6,Subject04_Session_1_EEG_RAW_V1,100656,69,6,6,6,5,46
7,Subject04_Session_2_EEG_RAW_V1,101243,72,6,6,6,6,48
8,Subject05_Session_1_EEG_RAW_V1,100566,72,6,6,6,6,48
9,Subject05_Session_1_Partial_low_battery...,108503,27,2,2,2,3,18
